# NexusFarm — model training

PlantVillage (38 classes) merged with a rice dataset (4 classes) = 42 classes.

**Before starting:** Runtime → Change runtime type → **T4 GPU** → Save.

Run the cells in order. Total time: about 50–70 minutes, mostly waiting.

### 1. Check the GPU
A `GPU` device should be listed. If the list is empty, set the runtime type again.

In [ ]:
import tensorflow as tf
print("TensorFlow", tf.__version__)
print(tf.config.list_physical_devices("GPU"))

### 2. Mount Google Drive
Results are written to Drive, so a disconnect does not lose the trained model.

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/NexusFarm_training"
os.makedirs(OUT, exist_ok=True)
print("Results will be saved in:", OUT)

### 3. Upload the Kaggle key
kaggle.com → Settings → API → **Create Legacy API Key**. Choose the downloaded `kaggle.json`.

In [ ]:
from google.colab import files
uploaded = files.upload()
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "wb") as f:
    f.write(uploaded["kaggle.json"])
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("Kaggle key ready")

### 4. Download both datasets
About 3 GB. Takes 5–10 minutes.

In [ ]:
!pip -q install kaggle
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p /content/data --unzip -q
!kaggle datasets download -d nirmalsankalana/rice-leaf-disease-image -p /content/data/rice --unzip -q
!ls /content/data

### 5. Upload the scripts
From the project's `backend/` folder, select all four at once:
`merge_datasets.py`, `train_full.py`, `evaluate.py`, `make_label_stubs.py`

In [ ]:
needed = {"merge_datasets.py", "train_full.py", "evaluate.py", "make_label_stubs.py"}
up = files.upload()
missing = needed - set(up)
print("Missing:", missing) if missing else print("All four scripts uploaded")

### 6. Merge the datasets
PlantVillage keeps the split it shipped with. Rice is split 70/15/15, grouping
duplicate and rotated images so no leaf appears on both sides of the split.

Check the output says **42 classes**.

In [ ]:
pv = None
for root, dirs, _ in os.walk("/content/data"):
    if "train" in dirs and "valid" in dirs and "rice" not in root:
        pv = root
        break
print("PlantVillage found at:", pv)
!rm -rf /content/merged
!python merge_datasets.py --plantvillage "{pv}" --rice /content/data/rice --out /content/merged

### 7. Train — 30 to 45 minutes
Keep this tab open and click the page every 15–20 minutes, or Colab will
disconnect for inactivity.

Write down the `Final validation accuracy` printed at the end.

In [ ]:
!python train_full.py --data_dir /content/merged --out "{OUT}/output"

### 8. Evaluate
First on the rice test split, which training never saw. Then on the full
validation set. Copy both outputs.

In [ ]:
!python evaluate.py --model "{OUT}/output/model.tflite" --labels "{OUT}/output/labels.txt" --data /content/merged/test --out "{OUT}/eval_test"

In [ ]:
!python evaluate.py --model "{OUT}/output/model.tflite" --labels "{OUT}/output/labels.txt" --data /content/merged/valid --out "{OUT}/eval_valid"

### 9. Generate the disease lookup for the app

In [ ]:
!python make_label_stubs.py --labels "{OUT}/output/labels.txt" --out "{OUT}/label_lookup_generated.dart"

### 10. Download everything

In [ ]:
!cd "{OUT}" && rm -f /content/nexusfarm_trained.zip && zip -rq /content/nexusfarm_trained.zip output/model.tflite output/labels.txt label_lookup_generated.dart eval_test eval_valid
files.download("/content/nexusfarm_trained.zip")

### Done
The zip holds `model.tflite`, `labels.txt`, `label_lookup_generated.dart` and the
evaluation CSVs. `FULL_DATASET.md` says where each file goes in the project.